# 00 — Setup & Data Exploration

Este notebook cobre:
1. Carregamento do ArtBench-10 (Kaggle local ou HuggingFace)
2. Subset de 20% fornecido pelo professor (`training_20_percent.csv`)
3. Visualização de amostras por estilo artístico
4. `HFDatasetTorch` — classe reutilizada em todos os outros notebooks

> **Dataset**: 50 000 imagens de treino · 32×32 RGB · 10 estilos artísticos

In [ ]:
from __future__ import annotations
import sys, random, csv
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

# ── paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT   = Path('..').resolve()
SCRIPTS_DIR = REPO_ROOT / 'TP' / 'TP1-alunos-src-only' / 'scripts'
SUBSET_CSV  = REPO_ROOT / 'TP' / 'TP1-alunos-src-only' / 'student_start_pack' / 'training_20_percent.csv'
KAGGLE_ROOT = REPO_ROOT / 'data' / 'ArtBench-10'   # local Kaggle download (optional)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── device ────────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():              return torch.device('cuda')
    if torch.backends.mps.is_available():      return torch.device('mps')
    return torch.device('cpu')

device = get_device()
print('Device:', device)

## 1. Carregar o dataset

In [ ]:
# ── opção A: Kaggle local ─────────────────────────────────────────────────────
# from artbench_local_dataset import load_kaggle_artbench10_splits
# hf_ds   = load_kaggle_artbench10_splits(KAGGLE_ROOT)

# ── opção B: HuggingFace (sem download prévio) ────────────────────────────────
from datasets import load_dataset
hf_ds = load_dataset('P1B3/artbench-10')

train_hf     = hf_ds['train']
class_names  = list(train_hf.features['label'].names)
num_classes  = len(class_names)

print('Train size :', len(train_hf))
print('Classes    :', class_names)

## 2. `HFDatasetTorch` — classe partilhada por todos os notebooks

Baseada no starter pack do professor. Aceita uma lista de índices para usar o subset de 20%.

In [ ]:
IMAGE_SIZE  = 32
BATCH_SIZE  = 64
USE_SUBSET  = True   # False → dataset completo

def safe_num_workers():
    return 0 if 'ipykernel' in sys.modules else 2


class HFDatasetTorch(Dataset):
    """Wrapper HuggingFace → PyTorch, com suporte a subset por índices."""
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds        = hf_split
        self.transform = transform
        self.indices   = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):  return len(self.indices)

    def __getitem__(self, idx):
        ex  = self.ds[self.indices[idx]]
        img = ex['image']
        y   = int(ex['label'])
        x   = self.transform(img) if self.transform else img
        return x, y


def load_ids_from_csv(csv_path: Path, column: str = 'train_id_original') -> list[int]:
    """Lê os índices do subset de 20% fornecido pelo professor."""
    ids = []
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            v = str(row.get(column, '')).strip()
            if v:
                ids.append(int(v))
    print(f'Loaded {len(ids)} ids from {csv_path.name}')
    return ids


def build_loader(hf_split, transform, indices=None, shuffle=True):
    ds = HFDatasetTorch(hf_split, transform, indices)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=safe_num_workers(), pin_memory=torch.cuda.is_available())


# normalização para [-1, 1]  (usada nos modelos generativos)
transform_norm = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# normalização para [0, 1]   (usada na avaliação FID/KID)
transform_01 = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
])

# ── construir loader ──────────────────────────────────────────────────────────
subset_ids   = load_ids_from_csv(SUBSET_CSV) if USE_SUBSET else None
train_loader = build_loader(train_hf, transform_norm, indices=subset_ids)

print(f'Amostras de treino : {len(train_loader.dataset)}')
print(f'Batches            : {len(train_loader)}')

## 3. Distribuição de classes

In [ ]:
labels = [train_hf[i]['label'] for i in (subset_ids if USE_SUBSET else range(len(train_hf)))]
counts = Counter(labels)

plt.figure(figsize=(10, 4))
plt.bar([class_names[k] for k in sorted(counts)], [counts[k] for k in sorted(counts)])
plt.xticks(rotation=30, ha='right')
plt.title('Amostras por estilo artístico' + (' (subset 20%)' if USE_SUBSET else ' (full)'))
plt.tight_layout()
plt.show()

## 4. Grelha de amostras

In [ ]:
x_batch, y_batch = next(iter(train_loader))
# denormalizar [-1,1] → [0,1]
x_show = (x_batch[:36] * 0.5 + 0.5).clamp(0, 1)

grid   = make_grid(x_show, nrow=6, padding=2)
plt.figure(figsize=(9, 9))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title('ArtBench-10 — amostras de treino')
plt.axis('off')
plt.show()

print('Labels:', [class_names[y] for y in y_batch[:36].tolist()])

## 5. Verificação de shape

In [ ]:
print('Batch shape :', x_batch.shape)       # (64, 3, 32, 32)
print('Pixel range :', x_batch.min().item(), '→', x_batch.max().item())  # [-1, 1]
print('Num classes :', num_classes)
print('Classes     :', class_names)